In [0]:
catalog = "meu_catalog"

silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print(f"Silver: {silver_schema}")
print(f"Gold: {gold_schema}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def chave_substituta(colunas_naturais):
    # Surrogate key: row_number() ordenado pela chave natural -> BIGINT determinístico.
    return F.row_number().over(Window.orderBy(*colunas_naturais)).cast("bigint")


def salvar_gold(df, nome_tabela):
    df.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(f"{gold_schema}.{nome_tabela}")
    print(f"{gold_schema}.{nome_tabela}: {spark.table(f'{gold_schema}.{nome_tabela}').count()} linhas")

In [0]:
dim_movies = (
    spark.table(f"{silver_schema}.tb_info_filmes")
    .select(
        chave_substituta(["id_filme"]).alias("sk_movie_id"),
        F.col("id_filme").cast("string"),
        F.col("titulo").cast("string"),
        F.col("data_lancamento").cast("date"),
        F.col("ano_lancamento").cast("int"),
        F.col("duracao_minutos").cast("int"),
        F.col("idioma_original").cast("string"),
        F.col("status_filme").cast("string"),
        F.col("sinopse").cast("string"),
    )
)

salvar_gold(dim_movies, "dim_movies")

In [0]:
pessoas_empresas = spark.table(f"{silver_schema}.tb_pessoas_empresas")

dim_genres = (
    spark.table(f"{silver_schema}.tb_generos")
    .select("nome_genero").distinct()
    .select(
        chave_substituta(["nome_genero"]).alias("sk_genre_id"),
        F.col("nome_genero").cast("string"),
    )
)

dim_people = (
    pessoas_empresas
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa"),
    )
    .distinct()
    .select(
        chave_substituta(["tipo_pessoa", "nome_pessoa"]).alias("sk_person_id"),
        F.col("nome_pessoa").cast("string"),
        F.col("tipo_pessoa").cast("string"),
    )
)

dim_companies = (
    pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .distinct()
    .select(
        chave_substituta(["nome_produtora"]).alias("sk_company_id"),
        F.col("nome_produtora").cast("string"),
    )
)

salvar_gold(dim_genres, "dim_genres")
salvar_gold(dim_people, "dim_people")
salvar_gold(dim_companies, "dim_companies")

In [0]:
financeiro = spark.table(f"{silver_schema}.tb_financeiro_filmes")
metricas = spark.table(f"{silver_schema}.tb_metricas_engajamento")

fact_movies_performance = (
    spark.table(f"{gold_schema}.dim_movies")
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")
    .join(financeiro, "id_filme", "left")
    .join(metricas, "id_filme", "left")
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int"),
    )
)

salvar_gold(fact_movies_performance, "fact_movies_performance")

In [0]:
dim_reviews = (
    spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"),
    )
    .join(spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "id_filme"), "id_filme")
    .select(
        chave_substituta(["sk_movie_id"]).alias("sk_review_id"),
        "sk_movie_id",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios",
    )
)

salvar_gold(dim_reviews, "dim_reviews")